# Lab 3b: Multi-Model Benchmark on Oxford Flowers-102

**Author:** Dr. Le Viet Duc, Hai-Long Nguyen

**Copyright:** © 2026 University of Twente. All rights reserved.

**License & Usage Terms:**
This notebook and its contents are provided strictly for educational purposes within the context of the **Intelligent Computing for Embedded Systems** course.

- **Permitted:** Students enrolled in this course may run, modify, and save a personal copy of this notebook for their own educational use and assignment submission.

- **Prohibited:** You may not distribute, publish, or share this material publicly. This includes, but is not limited to, uploading to public GitHub repositories, course-sharing websites (e.g., CourseHero, Chegg), or public Google Drive folders.

Unauthorized distribution constitutes an academic integrity violation and a breach of intellectual property rights.

---

### Lab Overview

**Duration:** 180 minutes  

**Follows:** Lecture 3 (Vision Models & Transfer Learning)  

**Platform:**

1. Your laptop with NVIDIA GPU

2. Conda 

**Group size:** 5

---

## Learning Objectives

**After this lab, you will be able to:**

1. Load and preprocess a real-world fine-grained image dataset (Oxford Flowers-102) with PyTorch

2. Adapt four pre-trained backbones (YOLO26n-cls, MobileNetV2, EfficientNet-B0, ResNet18) to a new 102-class problem via transfer learning

3. Write a reusable train / evaluate / fine-tune pipeline that works for arbitrary `nn.Module` classifiers — including non-torchvision models (Ultralytics)

4. Compare four models on the **same** dataset under the **same** training budget — accuracy, parameter count, training time

5. Export every trained model to **ONNX** ready for Lab 4 (HEF compilation + Hailo deployment)

---

### How to work through this notebook

Look for:

- `# TODO (...)` comments — each one tells you exactly what to implement and gives small hints.

- `raise NotImplementedError(...)` lines — **delete** them once you have written your code.

- `... # <-- your code here` placeholders — replace the `...` with a real expression.

You are expected to write the missing code yourself. The surrounding context (imports, variables, prints) is given so you can focus on the *interesting* part: the transforms, the transfer-learning head swap, the train/eval loops, the best-checkpoint logic, the ONNX export, and the NPZ test set builder.

Run the cells **top to bottom** — later cells depend on objects (`results`, `onnx_files`, …) defined earlier. If something goes wrong, fix it before moving on.

---


## Part A: Dataset Preparation (25 min)

### A1. Environment sanity check

We do a quick sanity check on PyTorch + CUDA before doing anything else. If `cuda` is `False`, stop and fix your environment — training on CPU is *very* slow for all four models.

In [1]:
import sys, platform, torch, torchvision

print(f"Python      : {sys.version.split()[0]}")
print(f"Platform    : {platform.platform()}")
print(f"PyTorch     : {torch.__version__}")
print(f"Torchvision : {torchvision.__version__}")
print(f"CUDA avail. : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device : {torch.cuda.get_device_name(0)}")
    print(f"CUDA capa.  : {torch.cuda.get_device_capability(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

Python      : 3.14.0
Platform    : macOS-26.4.1-arm64-arm-64bit-Mach-O
PyTorch     : 2.12.0
Torchvision : 0.27.0
CUDA avail. : False

Using device: cpu


### A2. Load the dataset, transforms and DataLoaders

`torchvision.datasets.Flowers102` downloads the official Oxford Flowers-102 dataset (~329 MB) automatically the first time you run this cell — no manual setup needed.

The dataset has three splits:

- `train`: 1 020 images (10 per class)
- `val`  : 1 020 images (10 per class)
- `test` : 6 149 images (the rest)

All four models we will use expect **224 × 224 RGB** inputs normalized with the **ImageNet mean/std** statistics, so we apply the same resize + normalize transform to every split.

In [2]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

IMG_SIZE      = 224
BATCH_SIZE    = 32
NUM_CLASSES   = 102
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# TODO (A2.1): Build the input transform pipeline.
#   All four backbones expect 224x224 RGB tensors normalized with the
#   ImageNet mean/std above. Use `transforms.Compose` with:
#     1. Resize to (IMG_SIZE, IMG_SIZE)
#     2. ToTensor (converts PIL -> [0,1] float tensor, CHW)
#     3. Normalize with IMAGENET_MEAN / IMAGENET_STD
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])  # <-- your code here

# TODO (A2.2): Create the three Flowers-102 splits.
#   Use `datasets.Flowers102` with root="data", split in {"train","val","test"},
#   transform=transform, download=True (first run downloads ~329 MB, then cached).
train_ds = datasets.Flowers102(root="data", split="train", transform=transform, download=True)
val_ds   = datasets.Flowers102(root="data", split="val", transform=transform, download=True)
test_ds  = datasets.Flowers102(root="data", split="test", transform=transform, download=True)

# TODO (A2.3): Wrap each split in a `DataLoader` with batch_size=BATCH_SIZE.
#   Shuffle ONLY the training loader.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds):>5d} samples")
print(f"Val  : {len(val_ds):>5d} samples")
print(f"Test : {len(test_ds):>5d} samples")
print(f"Classes : {NUM_CLASSES}")

100.0%
100.0%
100.0%


ModuleNotFoundError: No module named 'scipy'

### A3. Quick visualization

Plot a 4 × 4 grid of training images with their integer class labels. This is mostly a sanity check that the data pipeline works.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def denorm(img_tensor):
    """Reverse ImageNet normalization for display.

    TODO (A3): Given a CHW tensor that was normalized with IMAGENET_MEAN/STD,
    return a CHW tensor in [0, 1] suitable for `imshow`.

    Hint: reshape mean and std to shape (3, 1, 1) so broadcasting works, then
    compute  img * std + mean  and clamp to [0, 1].
    """
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    denormalized_img = img_tensor * std + mean
    return torch.clamp(denormalized_img, 0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img, lbl in zip(axes.flat, images[:16], labels[:16]):
    ax.imshow(denorm(img).permute(1, 2, 0).numpy())
    ax.set_title(f"class {lbl.item()}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Part B: Model Zoo Setup (20 min)

We will benchmark four **edge-oriented** backbones — three CNNs we keep from before, plus the canonical YOLO classifier that matters for Hailo deployment:

| Model           | Year | Family                                   | Why it is in this lab                                              |
|-----------------|------|------------------------------------------|--------------------------------------------------------------------|
| YOLO26n-cls     | 2025 | YOLO CNN classifier                      | Ultralytics' nano classifier — the canonical Hailo edge model      |
| MobileNetV2     | 2018 | Depthwise-separable CNN                  | Designed for mobile/edge (used in Labs 1, 2a, 2b)                  |
| EfficientNet-B0 | 2019 | Compound-scaled CNN                      | NAS-designed, strong accuracy/parameter trade-off                 |
| ResNet18        | 2015 | Residual CNN                             | The classic residual baseline — simple and very well supported on Hailo |

For each model we:

1. Load **pretrained** weights (torchvision `IMAGENET1K_V1` for MobileNetV2 / EfficientNet-B0 / ResNet18, Ultralytics for YOLO26n-cls)
2. Replace the classification head with a `Linear(in_features → 102)` layer
3. Feed it the **same** 224×224 ImageNet-normalized inputs as every other model, for a fair comparison

> **One of these is not a plain torchvision module:**
> - `YOLO("yolo26n-cls.pt").model` is an Ultralytics `ClassificationModel`. In eval mode its head returns `(probs, logits)`, so `build_yolo26n_cls` strips the softmax to expose plain logits for `CrossEntropyLoss` and ONNX export.
> - The other three (MobileNetV2, EfficientNet-B0, ResNet18) are standard torchvision backbones — we just swap the final `Linear` head to 102 classes.
> - All four are **pure-convolution CNNs** with no attention layers, so they export cleanly to ONNX **opset 13** (no fused `scaled_dot_product_attention` op that would force opset 14+) — exactly what the Hailo Dataflow Compiler wants in Lab 4.

In [3]:
import torch.nn as nn
import torchvision.models as tvm
from ultralytics import YOLO

# ---------------------------------------------------------------------------
# Transfer learning, the pattern:
#   1. Load a backbone with ImageNet-pretrained weights.
#   2. REPLACE its final classifier `Linear` layer with a new one whose
#      out_features == NUM_CLASSES (102).
#   3. Return the model. All other weights stay pretrained.
# ---------------------------------------------------------------------------

def build_yolo26n_cls(num_classes: int = NUM_CLASSES, weights: str = "yolo26n-cls.pt") -> nn.Module:
    # YOLO is given to you fully -- its head is non-standard (returns (probs, logits)
    # in eval mode). Study it as a reference, then implement the three torchvision
    # builders below by analogy.
    net = YOLO(weights).model
    for p in net.parameters():
        p.requires_grad_(True)
    head = net.model[-1]
    head.linear = nn.Linear(head.linear.in_features, num_classes)
    def _logits_only(x, _h=head):
        if isinstance(x, list):
            x = torch.cat(x, 1)
        return _h.linear(_h.drop(_h.pool(_h.conv(x)).flatten(1)))
    head.forward = _logits_only
    return net

def build_mobilenet_v2(num_classes: int = NUM_CLASSES) -> nn.Module:
    weights = tvm.MobileNet_V2_Weights.IMAGENET1K_V1
    m = tvm.mobilenet_v2(weights=weights)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m

def build_efficientnet_b0(num_classes: int = NUM_CLASSES) -> nn.Module:
    weights = tvm.EfficientNet_B0_Weights.IMAGENET1K_V1
    m = tvm.efficientnet_b0(weights=weights)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m

def build_resnet18(num_classes: int = NUM_CLASSES) -> nn.Module:
    weights = tvm.ResNet18_Weights.IMAGENET1K_V1
    m = tvm.resnet18(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

MODEL_BUILDERS = {
    "yolo26n_cls":     build_yolo26n_cls,
    "mobilenet_v2":    build_mobilenet_v2,
    "efficientnet_b0": build_efficientnet_b0,
    "resnet18":        build_resnet18,
}


NameError: name 'NUM_CLASSES' is not defined

### B1. Parameter and size comparison

Before training, look at the static cost of each model. Number of parameters is a first proxy for memory and (very roughly) FLOPs.

In [ ]:
def count_params(model: nn.Module):
    """Return (total_params, trainable_params) for `model`."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print(f"{'Model':<18}{'Total params':>15}{'Trainable':>15}{'Approx fp32 MB':>17}")
print("-" * 65)
for name, builder in MODEL_BUILDERS.items():
    m = builder()
    tot, tr = count_params(m)
    mb = tot * 4 / (1024 ** 2)            # 4 bytes per fp32 parameter
    print(f"{name:<18}{tot:>15,d}{tr:>15,d}{mb:>15.2f} MB")
    del m


## Part C: Training & Validation (90 min)

### C1. Generic training utilities

We define one pair of functions (`train_one_epoch`, `evaluate`) that works for **any** model in our zoo. This is the same pattern used in Lab 2b.

In [ ]:
import time
from tqdm.auto import tqdm

def train_one_epoch(model, loader, optimizer, criterion, device):
    """One pass over `loader` in training mode.

    TODO (C1.1): Implement the standard PyTorch training loop:
      1. `model.train()`
      2. For each (images, labels) batch:
         a. Move tensors to `device` (use `non_blocking=True`).
         b. Zero the optimizer gradients (`set_to_none=True` is preferred).
         c. Forward:    outputs = model(images)
         d. Loss:       loss = criterion(outputs, labels)
         e. Backward:   loss.backward()
         f. Step:       optimizer.step()
         g. Accumulate running loss (loss.item() * batch_size) and #correct
            (argmax of outputs == labels).
      3. Return (avg_loss, accuracy_percent).

    Tip: wrap the loader in `tqdm(loader, desc="train", leave=False)` so you
    get a progress bar.
    """
    raise NotImplementedError("Implement train_one_epoch()")

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """One evaluation pass over `loader` (no gradient updates).

    TODO (C1.2): Mirror `train_one_epoch` but:
      * call `model.eval()`
      * NO optimizer / backward / step
      * the `@torch.no_grad()` decorator already disables autograd.
    Return (avg_loss, accuracy_percent).
    """
    raise NotImplementedError("Implement evaluate()")


### C2. End-to-end training routine

`fit_model` ties everything together:

- builds the model, moves it to `device`
- creates an `AdamW` optimizer with a small learning rate (we are fine-tuning, not training from scratch)
- runs `EPOCHS` epochs, recording train/val loss and accuracy
- keeps a deep copy of the best validation-accuracy weights and reloads them at the end

> ⚠️ With only 1 020 training images the model overfits quickly — we use a **small** learning rate and a **small** number of epochs to keep the lab reproducible inside one session.

In [ ]:
import copy

EPOCHS = 10            # bump to 10–15 if you have time, try to choose most efficient settings
LR     = 1e-4
WD     = 1e-4

def fit_model(name: str, builder, train_loader, val_loader, device,
              epochs=EPOCHS, lr=LR, weight_decay=WD):
    model = builder().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_state = 0.0, None
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss);   history["val_acc"].append(vl_acc)
        print(f"[{name}] epoch {epoch:2d}/{epochs}  "
              f"train loss={tr_loss:.3f} acc={tr_acc:5.2f}%  |  "
              f"val loss={vl_loss:.3f} acc={vl_acc:5.2f}%")

        # TODO (C2): "Best-checkpoint" tracking.
        #   If `vl_acc` is the best validation accuracy seen so far, update
        #   `best_acc` AND take a deep copy of `model.state_dict()` into
        #   `best_state`. Use `copy.deepcopy` -- a plain reference would be
        #   overwritten by the next epoch's optimizer step.
        # >>> your code here <<<

    elapsed = time.time() - t0
    if best_state is not None:
        model.load_state_dict(best_state)         # restore best weights
    print(f"[{name}] best val acc = {best_acc:.2f}%   training time = {elapsed:.1f}s\n")
    return model, history, best_acc, elapsed


### C3. Train all four models

This cell trains the four models sequentially. Each model is trained from the same starting point (ImageNet weights) for the same number of epochs, with the same optimizer and the same data loaders — **a fair side-by-side comparison.**

You can modify the training hyperparameters (learning rate, number of epochs) to make the best version of each model.

In [ ]:
results = {}     # name -> dict(model, history, best_val_acc, train_time_s)

for name, builder in MODEL_BUILDERS.items():
    print(f"\n========== Training {name} ==========")
    model, history, best_val_acc, train_time = fit_model(
        name, builder, train_loader, val_loader, device)
    results[name] = {
        "model": model,
        "history": history,
        "best_val_acc": best_val_acc,
        "train_time_s": train_time,
    }

### C4. Training curves

Plot loss and accuracy curves for the four models on the same figure to visualise convergence speed.

In [ ]:
# TODO (C4): Plot training curves for all four models side-by-side.
#   Create a (1, 2) subplot figure. For each `name, r` in `results.items()`:
#     * left subplot : r["history"]["val_loss"] vs epoch
#     * right subplot: r["history"]["val_acc"]  vs epoch
#   Give each axes a title ("Validation loss", "Validation accuracy (%)"),
#   xlabel "epoch", a grid, and a legend with the model names.
#
# Hint: `axes[0].plot(history["val_loss"], marker="o", label=name)`

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# >>> your code here <<<
plt.tight_layout()
plt.show()


## Part D: Test Set Evaluation (15 min)

The Flowers-102 test set is **larger than train+val combined** (6 149 images), so it is a strong, mostly-unseen benchmark.

In [ ]:
criterion = nn.CrossEntropyLoss()

print(f"{'Model':<18}{'Best Val Acc':>15}{'Test Acc':>12}{'Train time (s)':>18}")
print("-" * 63)
for name, r in results.items():
    _, test_acc = evaluate(r["model"], test_loader, criterion, device)
    r["test_acc"] = test_acc
    print(f"{name:<18}{r['best_val_acc']:>14.2f}%{test_acc:>11.2f}%{r['train_time_s']:>17.1f}")

### Results Table

Fill in the table below from the output of the previous cell (re-run the notebook top-to-bottom — the numbers will vary slightly between runs).

| Model           | Params (M) | Best Val Acc (%) | Test Acc (%) | Train time (s) |
|-----------------|-----------:|-----------------:|-------------:|---------------:|
| YOLO26n-cls     |       ~1.7 |                  |              |                |
| MobileNetV2     |       ~2.3 |                  |              |                |
| EfficientNet-B0 |       ~4.1 |                  |              |                |
| ResNet18        |      ~11.2 |                  |              |                |

---

## Part E: Export to ONNX (20 min)

ONNX is the common interchange format we use throughout the course. In **Lab 3 / Lab 4** these `.onnx` files are compiled to `.hef` (Hailo Executable Format) so they can run on the **Hailo-10H** on the Raspberry Pi 5 AI HAT+.

We use the **same export settings as [`export_yolo26n_onnx.py`](export_yolo26n_onnx.py)** so the graphs feed straight into the Hailo Dataflow Compiler. Each model is:

1. moved to **CPU**, put in `eval()` mode, and exported in **fp32** (`half=False` — Hailo quantizes to int8 itself)
2. exported with a single example tensor of shape `(1, 3, 224, 224)` and a **static shape** (`dynamic=False` — the Hailo parser needs fixed dims, so no `dynamic_axes`)
3. exported with **opset 13** (all four models are pure CNNs with no attention ops, so opset 13 is sufficient — there is no fused `scaled_dot_product_attention` that would force opset 14+)
4. **simplified** (`simplify=True`) to fold constants the parser dislikes
5. quickly verified by loading it with `onnxruntime` and comparing the output to the PyTorch one

> **What comes next.** After Part E1 you will have four `.onnx` files under `week-4/onnx_models/`. Part **E2** then walks you through compiling each one to `.hef`, and Part **F** builds the `flowers102_test.npz` evaluation set that the *other* sub-group will use on the Raspberry Pi to measure accuracy of every `.hef`.


In [ ]:
from pathlib import Path

ONNX_DIR = Path("./onnx_models").resolve()
ONNX_DIR.mkdir(parents=True, exist_ok=True)

# Export settings mirror week-4/export_yolo26n_onnx.py so these .onnx files feed
# straight into the Hailo Dataflow Compiler (Lab 4 -> .hef on the AI HAT+):
#   * static shape (dynamic=False)  - the Hailo parser needs fixed dims
#   * simplified   (simplify=True)  - folds constants the parser dislikes
#   * fp32         (half=False)     - Hailo quantizes to int8 itself
# opset 13 with a 224x224 input (all four models are pure CNNs with no attention,
# so opset 13 is sufficient - there is no fused scaled_dot_product_attention op
# that would require opset 14+).

def export_onnx(name: str, model: nn.Module, out_dir: Path = ONNX_DIR,
                img_size: int = 224, opset: int = 11,
                simplify: bool = True) -> Path:
    model = model.eval().cpu()                            # CPU + eval, fp32
    dummy = torch.randn(1, 3, img_size, img_size)         # static (1, 3, 224, 224)
    out_path = out_dir / f"{name}_flowers102.onnx"
    torch.onnx.export(
        model, dummy, str(out_path),
        input_names=["input"], output_names=["logits"],
        opset_version=opset, do_constant_folding=True,    # static shape: no dynamic_axes
        dynamo=False,                                     # legacy exporter -> flat graph at the
    )                                                     # target opset. torch>=2.9 defaults to
                                                          # the dynamo path, which emits ONNX
                                                          # local functions that both the opset-13
                                                          # down-conversion and the Hailo parser
                                                          # reject; the flat graph avoids that.
    if simplify:                                          # fold constants the Hailo parser dislikes
        try:
            import onnx
            try:
                from onnxslim import slim                  # simplifier used by modern Ultralytics
                onnx.save(slim(onnx.load(str(out_path))), str(out_path))
            except ImportError:
                from onnxsim import simplify as onnx_simplify
                model_simp, ok = onnx_simplify(onnx.load(str(out_path)))
                if ok:
                    onnx.save(model_simp, str(out_path))
        except Exception as e:
            print(f"[{name}] simplify skipped ({type(e).__name__}: {e})")
    return out_path

onnx_files = {}
for name, r in results.items():
    p = export_onnx(name, r["model"])
    size_mb = p.stat().st_size / (1024 ** 2)
    onnx_files[name] = p
    print(f"[{name}] exported -> {p.name}  ({size_mb:.2f} MB)")

### E1. Verify ONNX outputs match PyTorch

A quick numerical check: feed the same random tensor through PyTorch and through an ONNX runtime, then compare. The max-absolute-difference should be on the order of `1e-4` or smaller.

The cell uses `onnxruntime` when it is available and otherwise falls back to onnx's built-in pure-NumPy `ReferenceEvaluator`, so it runs in any environment with no extra install or environment switch.

In [ ]:
# Run each exported ONNX and compare to PyTorch. We try onnxruntime first; if it is
# missing or incompatible with this environment's NumPy (onnxruntime 1.18.x is built
# for NumPy 1.x and will not import under NumPy 2), we transparently fall back to
# onnx's built-in pure-NumPy ReferenceEvaluator. So this cell runs top-to-bottom in
# any environment, with no extra install or environment switch.
def onnx_runner(path):
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
        return (lambda arr: sess.run(["logits"], {"input": arr})[0]), "onnxruntime"
    except Exception:
        from onnx.reference import ReferenceEvaluator
        ev = ReferenceEvaluator(str(path))
        return (lambda arr: ev.run(["logits"], {"input": arr})[0]), "onnx.reference"

x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

print(f"{'Model':<18}{'Max |Δ|':>12}{'Top-1 match':>14}{'Backend':>16}")
print("-" * 60)
for name, r in results.items():
    pt_out = r["model"].cpu().eval()(x).detach().numpy()
    run, backend = onnx_runner(onnx_files[name])
    ox_out = run(x.numpy())
    max_abs = float(np.max(np.abs(pt_out - ox_out)))
    same_top1 = bool(pt_out.argmax(1)[0] == ox_out.argmax(1)[0])
    print(f"{name:<18}{max_abs:>12.2e}{str(same_top1):>14}{backend:>16}")

### E2. Compile ONNX → HEF for Hailo-10H

> ⚙️ **Run on the compilation workstation**, *not* on the Raspberry Pi.
> Requires the **Hailo Dataflow Compiler 5.3.0** (see [`Guideline-HailoDataflowCompiler-Setup.md`](Guideline-HailoDataflowCompiler-Setup.md)) and the `exportyolov26n` conda environment.
> Full reference: [`HEF_CONVERSION_GUIDE.md`](HEF_CONVERSION_GUIDE.md).

The flow is *identical* for all four classifiers (`mobilenetv2`, `efficientnet_b0`, `resnet18`, `yolo26n_cls`) because they are pure CNNs with a 224×224×3 input.

**Step 1 — Capture a real calibration dataset on the Pi (this step should be done by other sub-groups - the one responsible for lab3_vision_pipeline_profiling.ipynb).** Random calibration data destroys INT8 accuracy. From the Raspberry Pi, point the Pi Camera at a few real flowers (or any natural scene with similar colour/texture statistics) and grab ~128 frames:

```bash
# On the Raspberry Pi 5
cd ~/

python3 capture_calib_dataset.py \
    --num-frames 128 \
    --imgsz 224 \
    --output calib_flowers.npy \
    --delay 0.1
```

Then copy the `.npy` back to the workstation:

```bash
# On the workstation
scp pi@<raspberrypi-ip>:~/calib_flowers.npy week-4/
```

**Step 2 — Compile each ONNX to HEF (workstation).**

```bash
conda activate exportyolov26n
cd week-4
mkdir -p hef_models

python3 convert_hef_file.py \
    --onnx <onnx-file-path> \
    --hef  <hef-file-path> \
    --hw-arch hailo10h \
    --calib-npy calib_flowers.npy
```

Key flags (see [`convert_hef_file.py`](convert_hef_file.py) for the full list):

| Flag | Meaning |
|---|---|
| `--hw-arch hailo10h` | **Required** for the Hailo-10H HAT+. Default in the script. |
| `--calib-npy <path>` | Real NHWC float32 calibration set from Step 1. |
| `--calib-shape N H W C` | Random calibration shape (fallback when `--calib-npy` is missing). For a 224 classifier use `100 224 224 3`. |
| `--end-node <name>` | Cut the graph here. Only needed for YOLO detection heads — **not** for the classifiers in this lab. |

**Step 3 — Sanity-check each HEF.**

```bash
hailortcli parse-hef hef_models/mobilenet_v2_flower102.hef | head
# the architecture string should mention hailo10h
```

**Step 4 — Hand off the four `.hef` files to the other sub-group.** They will copy them next to `flowers102_test.npz` (built in Part F below) on the Raspberry Pi and run `evaluate_accuracy.py`.


## Part F: Build the Evaluation Dataset for the NPU (15 min)

The `evaluate_accuracy.py` script that the other sub-group runs on the Raspberry Pi needs **one single file**: `flowers102_test.npz` containing the Oxford Flowers-102 *test* split as `uint8` NHWC RGB images plus the matching integer labels.

We build the `.npz` here on the workstation because we already have the dataset cached from Part A. Copy the resulting file to the Pi together with the four `.hef` files.

> **NPZ schema** (must match what `evaluate_accuracy.py` expects):
> - `images`  — `(N, 224, 224, 3)` `uint8`, RGB, values in `[0, 255]`
> - `labels`  — `(N,)` `int64`, class indices in `[0, 101]`


In [ ]:
import numpy as np
import torchvision.transforms as T
from torchvision.datasets import Flowers102

# Match the export resolution (224x224, RGB, uint8). NO normalisation here:
# the .hef expects raw uint8 in [0, 255] -- normalisation is folded into the
# quantised weights at compile time.
EVAL_SIZE = 224

# TODO (F.1): Build a transform that yields a 224x224 PIL image (NOT a tensor!):
#   Resize(256) -> CenterCrop(EVAL_SIZE).  Do NOT include ToTensor / Normalize.
to_uint8_nhwc = ...

test_ds = Flowers102(root='../data', split='test', download=True, transform=to_uint8_nhwc)
print(f"Test set: {len(test_ds)} images")

# TODO (F.2): Allocate the output arrays:
#   images: shape (N, 224, 224, 3), dtype uint8
#   labels: shape (N,),             dtype int64
images = ...
labels = ...

# TODO (F.3): Fill `images` and `labels` by iterating over `test_ds`.
#   Each `img` is a PIL RGB image -> use `np.asarray(img, dtype=np.uint8)`
#   which yields HWC uint8 directly. Convert `lbl` with `int(lbl)`.
for i, (img, lbl) in enumerate(test_ds):
    ...  # >>> your code here <<<

out_path = 'flowers102_test.npz'
np.savez_compressed(out_path, images=images, labels=labels)
print(f"Saved {out_path}: images={images.shape} {images.dtype}, "
      f"labels={labels.shape} {labels.dtype}")


### F1. Hand-off to the Raspberry Pi sub-group

Copy **both** the four HEFs and the evaluation set to the Pi:

```bash
# On the workstation
scp week-4/flowers102_test.npz pi@<raspberrypi-ip>:~/
scp week-4/hef_models/*.hef    pi@<raspberrypi-ip>:~/
```

On the Pi, the other sub-group will then run (see Lab 3, Part C1):

```bash
python3 evaluate_accuracy.py \
    --model <hef-file-path> \
    --test-data flowers102_test.npz \
    --output-csv c1_accuracy.csv
```

The resulting `c1_accuracy.csv` (one row per model) goes into the C1 accuracy column of the assignment table.
